# Notebook 16 — Statistics on Expression Data: End-to-End Pipeline

**Module:** 03 — Statistics and Probability  
**Notebook:** 16 of 18  
**Prerequisites:** NB02, NB06, NB09, NB11, NB12  
**Time estimate:** 90 minutes

> **Integration notebook:** This notebook integrates the full Module 03 toolkit —
> normalization → testing → correction → visualization — on a realistic RNA-seq
> dataset simulation. It is the bridge to NB17's real-data mini-project.

---
## Step 1 — Motivation

A Track A bioinformatics interview will hand you a count matrix and ask: 'walk me
through your analysis.' This notebook is the answer — every step, every decision,
every pitfall explained from first principles.

---
## Step 3 — Biological Background

**RNA-seq count data:**
- Raw counts are integers: number of reads mapping to each gene.
- Counts are **not** normally distributed — they are discrete, right-skewed, and
  their variance scales with the mean (unlike normal data).
- **Normalization is mandatory** before comparison: different samples have different
  library sizes (total reads).

**Standard normalization methods:**
- **CPM (Counts Per Million):** divide each gene's count by total library size × 1e6.
  Corrects for library size but not gene length.
- **TPM (Transcripts Per Million):** correct for both library size and gene length.
  Allows comparison within and between samples.
- **Log2(CPM + 1):** log-transforms CPM for downstream statistical analysis.
  Approximately normal for highly expressed genes.

**DESeq2's size factor normalization:** estimates a scaling factor per sample
using the geometric mean of all genes — more robust to composition effects.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
# Cell 6.1 — Simulate realistic RNA-seq count matrix
rng = np.random.default_rng(42)

n_genes  = 2000
n_ctrl   = 6
n_trt    = 6
n_samples = n_ctrl + n_trt
n_de     = 200   # truly differentially expressed genes

# Gene-specific mean expression levels (highly variable)
base_means = np.exp(rng.normal(4, 2, n_genes))   # log-normal baseline

# Library sizes: realistic 10M–30M reads per sample
lib_sizes = rng.integers(10_000_000, 30_000_000, n_samples)

# True DE gene indices and effect sizes
de_genes = rng.choice(n_genes, n_de, replace=False)
de_effects = rng.choice([-1, 1], n_de) * rng.uniform(0.5, 2.5, n_de)  # log2FC
true_log2fc = np.zeros(n_genes)
true_log2fc[de_genes] = de_effects

# Simulate counts: Negative Binomial with overdispersion
dispersion = 0.1  # NB dispersion parameter
counts = np.zeros((n_genes, n_samples), dtype=int)
for s in range(n_samples):
    is_trt = s >= n_ctrl
    size_factor = lib_sizes[s] / lib_sizes.mean()
    for g in range(n_genes):
        mu = base_means[g] * size_factor
        if is_trt:
            mu *= 2 ** true_log2fc[g]
        # NB: mean=mu, var=mu + mu^2*dispersion
        p = 1 / (1 + mu * dispersion)
        r = 1 / dispersion
        counts[g, s] = rng.negative_binomial(r, p)

sample_labels = ['ctrl'] * n_ctrl + ['trt'] * n_trt
print(f"Count matrix shape: {counts.shape} (genes × samples)")
print(f"Library sizes: {lib_sizes // 1_000_000} M reads")
print(f"Count sparsity: {(counts == 0).mean():.2%} zeros")

In [ ]:
# Cell 6.2 — Normalization: CPM and log2(CPM+1)
def counts_to_cpm(counts: np.ndarray) -> np.ndarray:
    """Counts Per Million normalization. counts: (genes, samples)"""
    lib_sizes = counts.sum(axis=0, keepdims=True)
    return counts / lib_sizes * 1e6

def log2cpm(counts: np.ndarray, pseudo: float = 1.0) -> np.ndarray:
    return np.log2(counts_to_cpm(counts) + pseudo)

lcpm = log2cpm(counts)

# Filter low-expression genes: keep genes with median log2CPM > 1 in at least one group
ctrl_median  = np.median(lcpm[:, :n_ctrl], axis=1)
trt_median   = np.median(lcpm[:, n_ctrl:], axis=1)
expressed = (ctrl_median > 1) | (trt_median > 1)

lcpm_filt = lcpm[expressed]
true_log2fc_filt = true_log2fc[expressed]
de_mask_filt = np.isin(np.where(expressed)[0], de_genes)

print(f"Genes retained after expression filter: {expressed.sum()} / {n_genes}")

In [ ]:
# Cell 6.3 — Differential expression: Welch t-test + BH FDR correction
def de_analysis(lcpm: np.ndarray, n_ctrl: int, alpha_fdr: float = 0.05) -> dict:
    """Full DE pipeline: t-test → BH correction → results table."""
    n_genes_filt = lcpm.shape[0]
    ctrl = lcpm[:, :n_ctrl]
    trt  = lcpm[:, n_ctrl:]

    log2fc = trt.mean(axis=1) - ctrl.mean(axis=1)
    p_vals = np.array([
        stats.ttest_ind(ctrl[g], trt[g], equal_var=False)[1]
        for g in range(n_genes_filt)
    ])

    _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')

    sig_mask = q_vals < alpha_fdr
    return dict(log2fc=log2fc, p_vals=p_vals, q_vals=q_vals,
                sig_mask=sig_mask, n_sig=sig_mask.sum())

de = de_analysis(lcpm_filt, n_ctrl)

# Evaluate performance
true_pos = (de['sig_mask'] & de_mask_filt).sum()
false_pos = (de['sig_mask'] & ~de_mask_filt).sum()
true_neg  = (~de['sig_mask'] & ~de_mask_filt).sum()
false_neg = (~de['sig_mask'] & de_mask_filt).sum()

print(f"DE analysis (FDR 5%):")
print(f"  Significant genes: {de['n_sig']}")
print(f"  True positives:    {true_pos}")
print(f"  False positives:   {false_pos}")
print(f"  Sensitivity:       {true_pos / de_mask_filt.sum():.2%}")
print(f"  Estimated FDR:     {false_pos / max(de['n_sig'], 1):.2%}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Full expression analysis figure panel
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.35)

# --- Panel 1: Library size before / after normalization ---
ax = fig.add_subplot(gs[0, 0])
sample_names = [f"C{i+1}" for i in range(n_ctrl)] + [f"T{i+1}" for i in range(n_trt)]
colors_bar = ["steelblue"] * n_ctrl + ["tomato"] * n_trt
ax.bar(sample_names, lib_sizes / 1e6, color=colors_bar, alpha=0.8)
ax.set_ylabel("Library size (M reads)")
ax.set_title("Library sizes (pre-normalization)")
ax.tick_params(axis='x', rotation=45)

# --- Panel 2: log2CPM density per sample ---
ax = fig.add_subplot(gs[0, 1])
for s in range(n_samples):
    color = "steelblue" if s < n_ctrl else "tomato"
    kde_x = np.linspace(-2, 15, 300)
    kde = stats.gaussian_kde(lcpm_filt[:, s])
    ax.plot(kde_x, kde(kde_x), color=color, alpha=0.6, lw=1)
ax.set_xlabel("log₂(CPM+1)"); ax.set_ylabel("Density")
ax.set_title("Normalized expression distributions\n(blue=ctrl, red=trt)")

# --- Panel 3: MA plot (mean vs. log2FC) ---
ax = fig.add_subplot(gs[0, 2])
mean_expr = lcpm_filt.mean(axis=1)
ax.scatter(mean_expr[~de['sig_mask']], de['log2fc'][~de['sig_mask']],
           s=4, alpha=0.3, color="gray", label="ns")
ax.scatter(mean_expr[de['sig_mask'] & de_mask_filt], de['log2fc'][de['sig_mask'] & de_mask_filt],
           s=10, alpha=0.8, color="steelblue", label="TP")
ax.scatter(mean_expr[de['sig_mask'] & ~de_mask_filt], de['log2fc'][de['sig_mask'] & ~de_mask_filt],
           s=10, alpha=0.8, color="tomato", label="FP")
ax.axhline(0, color='black', lw=0.5, ls='--')
ax.set_xlabel("Mean log₂(CPM+1)"); ax.set_ylabel("log₂FC")
ax.set_title("MA plot"); ax.legend(fontsize=7)

# --- Panel 4: Volcano plot ---
ax = fig.add_subplot(gs[1, 0])
neg_log10q = -np.log10(de['q_vals'] + 1e-300)
ax.scatter(de['log2fc'][~de['sig_mask']], neg_log10q[~de['sig_mask']],
           s=4, alpha=0.3, color="gray")
ax.scatter(de['log2fc'][de['sig_mask'] & de_mask_filt], neg_log10q[de['sig_mask'] & de_mask_filt],
           s=10, alpha=0.8, color="steelblue", label="TP")
ax.scatter(de['log2fc'][de['sig_mask'] & ~de_mask_filt], neg_log10q[de['sig_mask'] & ~de_mask_filt],
           s=10, alpha=0.8, color="tomato", label="FP")
ax.axhline(-np.log10(0.05), color='black', ls='--', lw=0.8)
ax.set_xlabel("log₂FC"); ax.set_ylabel("-log₁₀(q)")
ax.set_title("Volcano plot (BH-corrected)"); ax.legend(fontsize=7)

# --- Panel 5: p-value histogram ---
ax = fig.add_subplot(gs[1, 1])
ax.hist(de['p_vals'], bins=40, color="steelblue", edgecolor="none", density=True)
ax.axhline(1.0, color="gray", ls="--", lw=1, label="Uniform null")
ax.set_xlabel("raw p-value"); ax.set_ylabel("Density")
ax.set_title("p-value histogram (anti-conservative spike = DE signal)")
ax.legend(fontsize=8)

# --- Panel 6: Heatmap of top 30 DE genes ---
ax = fig.add_subplot(gs[1, 2])
top_idx = np.argsort(de['q_vals'])[:30]
heatmap_data = lcpm_filt[top_idx]
heatmap_scaled = (heatmap_data - heatmap_data.mean(axis=1, keepdims=True)) / (
    heatmap_data.std(axis=1, keepdims=True) + 1e-9)
im = ax.imshow(heatmap_scaled, aspect='auto', cmap='RdBu_r', vmin=-2, vmax=2)
ax.set_xticks(range(n_samples))
ax.set_xticklabels(sample_names, rotation=45, fontsize=7)
ax.set_ylabel("Top 30 DE genes"); ax.set_yticks([])
ax.set_title("Top 30 DE genes (z-scored)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Z-score")

plt.suptitle("Module 03 — End-to-end RNA-seq Statistical Analysis",
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig("../../../portfolio/module03_expression_analysis_overview.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved to portfolio/module03_expression_analysis_overview.png")

---
## Step 8 — Exercises

1. Modify `de_analysis()` to use a log-fold-change threshold in addition to FDR:
   a gene is significant only if q < 0.05 AND |log2FC| > 1. How does this change
   sensitivity and false discovery rate?
2. Implement `counts_to_cpm()` and `log2cpm()` from scratch. Verify that after
   CPM normalization all columns sum to 1,000,000.
3. The MA plot shows a relationship between mean expression and effect size reliability.
   What pattern do you see for low-expression genes? Why does this occur?
4. What is 'composition bias' in RNA-seq normalization? When would CPM fail but
   DESeq2's size factor method succeed? Give a concrete example.

---
## Quiz — Active Recall (Track A critical)

1. What does CPM normalization correct for? What does it NOT correct for?
2. Walk through the DE analysis pipeline in this notebook: 5 steps, 30 seconds each.
3. What does an MA plot show? What does the name 'MA' stand for?
4. Why are raw RNA-seq counts not normally distributed?
5. What is the minimum number of biological replicates for a DE analysis? Why?

---
## Reflection

**Date completed:** ____________________

> *[Could you walk an interviewer through a DE analysis step by step, explaining every decision? Which step feels shakiest?]*

---
*Next: `17_mini_project_statistical_report.ipynb`*